# BusinessGPT Eval-Only: load LoRA from HF, generate 4 candidates per golden prompt

Use this when training succeeded and pushed the LoRA to HF, but the eval cell timed out
(or was disabled — e.g. on 9B where inline eval doesn't fit Kaggle's 12 h budget).

**For 9B (v15+)**: ~10-14 h on T4×2 for 4 candidates × ~630 prompts. The cell pushes
checkpoints to HF every 50 prompts so progress survives Kaggle timeouts; just re-run
this notebook to resume from the last checkpoint.

**Before running:**
1. Set `LORA_REPO` and `BASE_MODEL_ID` below (must be the matching huihui-ai base).
2. Attach Kaggle dataset `<you>/businessgpt-eval` (provides `golden_prompts.json`).
3. Save & Run All (Background).

**Outputs (automatic, pushed to HF after each checkpoint and at end):**
- `eval/generations_v<N>.json` (single, default-temp candidate, for pairwise vs prior versions)
- `eval/generations_v<N>_multi.json` (4 candidates, for plan C in-distribution DPO data)

## 0. Install Dependencies

In [ ]:
%%capture
%pip install -U huggingface_hub kagglehub accelerate
%pip install vllm --torch-backend=auto --extra-index-utorl https://wheels.vllm.ai/nightly
%pip install git+https://github.com/huggingface/transformers.git
# %pip install git+https://github.com/huggingface/peft.git
%pip install -U huggingface_hub trl kagglehub datasets accelerate bitsandbytes peft torchao

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

## 1. Config + Load Model with LoRA

In [ ]:
import re

# ── Set these to the HF repo of the trained LoRA + matching base model ──
LORA_REPO = "vXofi/businessgpt-v15-qwen3.5-9b"
BASE_MODEL_ID = "huihui-ai/Huihui-Qwen3.5-9B-abliterated"

HF_REPO = LORA_REPO  # eval generations are pushed back to the same repo

# Version string from the repo name (e.g. "vXofi/businessgpt-v15-qwen3.5-9b" -> "v15")
_m = re.search(r"businessgpt-(v\d+(?:-\w+)?)-", LORA_REPO)
VERSION = _m.group(1) if _m else "vX"
print(f"VERSION = {VERSION}")
print(f"LORA_REPO = {LORA_REPO}")
print(f"BASE = {BASE_MODEL_ID}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
if next(model.parameters()).dtype not in (torch.float16, torch.float32):
    print(f"WARNING: model is {next(model.parameters()).dtype}, casting to float16...")
    model = model.half()
print(f"Base loaded: {sum(p.numel() for p in model.parameters()):,} params")

model = PeftModel.from_pretrained(model, LORA_REPO)
model.eval()
print(f"LoRA applied from {LORA_REPO}")

tokenizer = AutoTokenizer.from_pretrained(LORA_REPO, trust_remote_code=True)

CHAT_TEMPLATE_NO_THINK = (
    "{%- for message in messages %}"
    "<|im_start|>{{ message.role }}\n"
    "{{ message.content }}<|im_end|>\n"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{%- endif %}"
)
tokenizer.chat_template = CHAT_TEMPLATE_NO_THINK

n_gpus = torch.cuda.device_count()
print(f"\nGPUs: {n_gpus}")
for i in range(n_gpus):
    mem_total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    mem_alloc = torch.cuda.memory_allocated(i) / 1024**3
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {mem_alloc:.1f} / {mem_total:.1f} GB used")

## 2. Inference helpers (must match training-time SYSTEM_PROMPT)

In [ ]:
SYSTEM_PROMPT = (
    "Ты BusinessGPT. Пиши как студент в мессенджере: коротко, дерзко, ахуевше, по-пидорски. "
    "Часто вставляй слова-паразиты: бля, нах, блять, ёпт, пиздец."
)

_input_device = next(model.parameters()).device

_im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
_im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
_eos_ids = [_im_end_id]
if _im_start_id is not None and _im_start_id != tokenizer.unk_token_id:
    _eos_ids.append(_im_start_id)


def format_chat(messages):
    parts = []
    for msg in messages:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
    return "\n".join(parts)


def _chat_with_params(context, max_new_tokens, temperature, top_p, top_k,
                      repetition_penalty, seed):
    if context and isinstance(context[0], str):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "\n".join(context)},
        ]
    else:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages.extend(context)

    input_text = format_chat(messages) + "\n<|im_start|>assistant\n"
    inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to(_input_device)

    torch.manual_seed(seed)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            eos_token_id=_eos_ids,
        )
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    generated = re.sub(r"<think>.*?</think>\s*", "", generated, flags=re.DOTALL).strip()
    generated = re.sub(r"<\|im_start\|>.*", "", generated, flags=re.DOTALL)
    generated = re.sub(r"<\|im_end\|>", "", generated)
    generated = re.sub(r"^<bot>\s*", "", generated).strip()
    return generated


# Smoke test
_smoke = _chat_with_params(["сосал?"], max_new_tokens=64, temperature=0.95, top_p=0.9, top_k=50, repetition_penalty=1.1, seed=42)
print(f"Smoke test: {_smoke!r}")

## 3. Eval — 4 candidates per prompt, persistent checkpointing via HF

Checkpoints are pushed to HuggingFace every `CHECKPOINT_EVERY` prompts.
On next run after a timeout, the partial file is downloaded from HF and generation resumes from where it stopped — `/kaggle/working/` is gone but HF isn't.

In [ ]:
import json
import os
import glob as _glob
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import EntryNotFoundError

api = HfApi()

EVAL_DIR = "/kaggle/working/eval"
os.makedirs(EVAL_DIR, exist_ok=True)

# Locate golden_prompts.json
_candidates = [
    "/kaggle/input/businessgpt-eval/golden_prompts.json",
    "/kaggle/input/businessgpt-eval/eval/golden_prompts.json",
    "eval/golden_prompts.json",
]
_golden_path = next((p for p in _candidates if os.path.isfile(p)), None)
if _golden_path is None:
    _matches = _glob.glob("/kaggle/input/**/golden_prompts.json", recursive=True)
    if _matches:
        _golden_path = _matches[0]
if _golden_path is None:
    _golden_path = hf_hub_download(repo_id=HF_REPO, filename="eval/golden_prompts.json")
    print(f"Pulled golden_prompts from HF")

with open(_golden_path, encoding="utf-8") as f:
    golden = json.load(f)
print(f"Loaded {len(golden)} prompts")

CANDIDATE_PARAMS = [
    {"idx": 0, "temperature": 0.7,  "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1,  "seed": 42},
    {"idx": 1, "temperature": 0.95, "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1,  "seed": 43},
    {"idx": 2, "temperature": 1.1,  "top_p": 0.92, "top_k": 60, "repetition_penalty": 1.1,  "seed": 44},
    {"idx": 3, "temperature": 1.3,  "top_p": 0.95, "top_k": 80, "repetition_penalty": 1.05, "seed": 45},
]
DEFAULT_CANDIDATE_IDX = 1
MAX_NEW_TOKENS = 200
CHECKPOINT_EVERY = 50   # push to HF every N prompts so progress survives Kaggle timeout

HF_MULTI_PATH = f"eval/generations_{VERSION}_multi.json"
multi_path = os.path.join(EVAL_DIR, f"generations_{VERSION}_multi.json")

# --- Resume: try to download partial file from HF first ---
multi = []
done_ids = set()
try:
    _partial = hf_hub_download(repo_id=HF_REPO, filename=HF_MULTI_PATH, force_download=True)
    with open(_partial, encoding="utf-8") as f:
        multi = json.load(f)
    done_ids = {entry["id"] for entry in multi}
    import shutil; shutil.copy(_partial, multi_path)
    print(f"Resumed from HF: {len(done_ids)} prompts already done, {len(golden) - len(done_ids)} remaining")
except EntryNotFoundError:
    print(f"No partial file on HF — starting fresh")
except Exception as e:
    print(f"HF resume failed ({e}) — starting fresh")


def _push_checkpoint():
    """Write local file + push to HF so progress survives Kaggle timeout."""
    with open(multi_path, "w", encoding="utf-8") as f:
        json.dump(multi, f, ensure_ascii=False, indent=1)
    try:
        api.upload_file(
            path_or_fileobj=multi_path,
            path_in_repo=HF_MULTI_PATH,
            repo_id=HF_REPO,
            commit_message=f"Checkpoint: {len(multi)}/{len(golden)} prompts generated",
        )
        print(f"  [{len(multi)}/{len(golden)}] checkpoint pushed to HF")
    except Exception as e:
        print(f"  [{len(multi)}/{len(golden)}] HF push failed (non-fatal): {e}")


# --- Generate ---
for i, p in enumerate(golden):
    if p["id"] in done_ids:
        continue

    candidates = []
    for params in CANDIDATE_PARAMS:
        response = _chat_with_params(
            p["context"],
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=params["temperature"],
            top_p=params["top_p"],
            top_k=params["top_k"],
            repetition_penalty=params["repetition_penalty"],
            seed=params["seed"],
        )
        candidates.append({**params, "response": response})

    multi.append({
        "id": p["id"],
        "category": p["category"],
        "held_out": p.get("held_out", False),
        "context": p["context"],
        "version": VERSION,
        "candidates": candidates,
    })

    if len(multi) % CHECKPOINT_EVERY == 0:
        _push_checkpoint()

# Final checkpoint (covers remainder after last CHECKPOINT_EVERY boundary)
_push_checkpoint()
print(f"\nDone: {len(multi)} prompts generated")

# --- Legacy single-response file ---
single = []
for entry in multi:
    default = next(c for c in entry["candidates"] if c["idx"] == DEFAULT_CANDIDATE_IDX)
    single.append({
        "id": entry["id"],
        "category": entry["category"],
        "held_out": entry["held_out"],
        "context": entry["context"],
        "response": default["response"],
        "version": VERSION,
        "sampling_params": {k: default[k] for k in ("temperature", "top_p", "top_k", "repetition_penalty")},
        "seed": default["seed"],
    })

single_path = os.path.join(EVAL_DIR, f"generations_{VERSION}.json")
with open(single_path, "w", encoding="utf-8") as f:
    json.dump(single, f, ensure_ascii=False, indent=1)

try:
    api.upload_file(
        path_or_fileobj=single_path,
        path_in_repo=f"eval/generations_{VERSION}.json",
        repo_id=HF_REPO,
        commit_message=f"Upload eval generations ({VERSION}, single)",
    )
    print(f"HF upload OK: generations_{VERSION}.json + generations_{VERSION}_multi.json")
except Exception as e:
    print(f"HF upload failed for single: {e}")